# 03 Tree models across F0, F1 and F2

Step 9. Run `.venv/bin/python -m src.train_trees` first. This notebook only reads the experiment log and selected configurations; it never fits models. The V-column list remains provisional.

Time CV uses seed 0: 20 LightGBM configurations per feature set, four Random Forest configurations, and four logistic-regression C values. Each chosen configuration is evaluated on validation with seeds 0, 1 and 2. LightGBM uses average-precision early stopping on the fold validation part (or the main validation block during the final fit). Thresholds are also selected on validation, so its precision/recall estimates are optimistic. Test scoring is reserved until experiment freeze.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(root))
import json
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data import ROOT
from src.experiment import logged_run
from src.results import LOG_PATH

log = pd.read_csv(LOG_PATH)
config = yaml.safe_load((ROOT / 'config' / 'best_params.yaml').read_text())
models = ['logreg', 'rf', 'lgbm']
feature_sets = ['F0', 'F1', 'F2']
rows = []
for model in models:
    for feature_set in feature_sets:
        params = config['models'][model][feature_set]['params']
        for seed in (0, 1, 2):
            row = logged_run(model, feature_set, 'val', seed, params)
            assert row is not None, (model, feature_set, seed)
            rows.append(row)
val = pd.DataFrame(rows)
assert len(val) == 27
assert not log['split'].eq('test').any(), 'test block must stay unscored before freeze'
print(f'{len(log)} logged fits; {len(val)} selected validation runs; no test scoring')

## Validation metrics (mean and sample standard deviation over three seeds)

In [ ]:
summary = val.groupby(['model', 'feature_set']).agg(
    pr_auc_mean=('pr_auc', 'mean'), pr_auc_std=('pr_auc', 'std'),
    auroc_mean=('auroc', 'mean'), auroc_std=('auroc', 'std'),
    recall_top1pct_mean=('recall_top1pct', 'mean'),
    recall_top1pct_std=('recall_top1pct', 'std'),
    runtime_s_mean=('runtime_s', 'mean'), runtime_s_min=('runtime_s', 'min'),
    runtime_s_max=('runtime_s', 'max'), seeds=('seed', 'nunique'))
assert summary['seeds'].eq(3).all()
summary.round(6)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(feature_sets))
width = 0.24
labels = {'logreg': 'Logistic regression', 'rf': 'Random Forest', 'lgbm': 'LightGBM'}
for i, model in enumerate(models):
    subset = summary.loc[model].reindex(feature_sets)
    ax.bar(x + (i - 1) * width, subset.pr_auc_mean, width,
           yerr=subset.pr_auc_std, capsize=4, label=labels[model])
ax.set_xticks(x, feature_sets)
ax.set_ylim(0, 1)
ax.set_ylabel('Validation PR-AUC')
ax.set_title('Step 9: model and history-feature comparison (provisional V list)')
ax.legend(title='Model')
fig.tight_layout()
path = ROOT / 'reports' / 'figures' / 'trees_validation.png'
path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(path, dpi=150)
plt.show()

## Selected configurations and per-fit outcomes

`params._fit` records LightGBM best iteration and its training-only class weight; `threshold` was selected on validation.

In [ ]:
val[['model', 'feature_set', 'seed', 'params', 'pr_auc', 'auroc', 'precision', 'recall', 'recall_top1pct', 'runtime_s']]

## Time-CV search results

Each row below is one candidate averaged across the four time folds. Fold-to-fold variation measures temporal variation; it differs from the seed variation above.

In [ ]:
cv = log[(log.split == 'cv') & (log.cv_scheme == 'time') & (log.seed == 0)
         & log.model.isin(models) & log.feature_set.isin(feature_sets)].copy()
cv['candidate'] = cv.params.map(lambda p: json.dumps(
    {k: v for k, v in json.loads(p).items() if k not in ('fold', '_fit')}, sort_keys=True))
cv_summary = cv.groupby(['model', 'feature_set', 'candidate']).agg(
    pr_auc_mean=('pr_auc', 'mean'), pr_auc_std=('pr_auc', 'std'),
    runtime_s_mean=('runtime_s', 'mean'), folds=('pr_auc', 'count'))
assert cv_summary.folds.eq(4).all()
assert len(cv_summary) == 3 * (20 + 4 + 4)
cv_summary.round(6)